# Period-scan sensitivity in the logistic map

This notebook is the slower companion to `reports/period-scan-sensitivity.md`.

The narrow question is the useful one: **how much of the stable-window picture disappears when the tail is too short?**


## 1. Hold the grid fixed and only change scan patience

To keep the comparison honest, this pass keeps the same `r` interval, sample count, and tolerance.
Only two patience knobs change:

- short scan: `warmup = 1200`, `keep = 64`
- deep scan: `warmup = 3600`, `keep = 512`

That way, a disagreement is easier to read as a cutoff effect instead of a moving-target setup change.


In [ ]:
from collections import Counter
from logisticlab.core import compare_period_scans

rows = compare_period_scans(3.0, 4.0, samples=400, short_warmup=1200, short_keep=64, deep_warmup=3600, deep_keep=512, max_period=64, tol=1e-8)
summary = {
    'recovered_stable': sum(1 for row in rows if row.recovered_stable),
    'lost_stable': sum(1 for row in rows if row.lost_stable),
    'shifted_period': sum(1 for row in rows if row.shifted_period),
}
summary


The first useful fact is qualitative, not numerical: the deeper scan should mostly **recover** stable windows, not invent a different global picture.

If the short scan were already enough, the recovered count would collapse toward zero.


In [ ]:
short_counts = Counter(row.short_row.detected_period for row in rows)
deep_counts = Counter(row.deep_row.detected_period for row in rows)
{
    'short': dict(short_counts),
    'deep': dict(deep_counts),
}


## 2. Where the cutoff bites first

The full-axis strips are useful, but the real action lives near chaos onset.
That is where narrow windows start to slip through a short scan first.


In [ ]:
interesting = [
    (round(row.r, 6), row.short_row.detected_period, row.deep_row.detected_period)
    for row in rows
    if 3.44 <= row.r <= 3.58 and row.changed
]
interesting[:20]


That table is the notebook version of the figure caption:

- wide low-period windows survive both scans
- narrow windows are what the short scan misses first
- some disagreements are not just chaos-versus-stable flips; the deeper scan can also resolve a higher period

So the thin-window story is not decorative. It is a real measurement sensitivity inside the public artifact.
